<a href="https://colab.research.google.com/github/AzureSolo/Land-suitability-project/blob/main/2_Raw_Data_splits_for_shareability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Run this cell with a Python runtime only.
# Switch back to R before running any cell below.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#switch runtime to R
setwd("/content/drive/MyDrive/land_suitability_project")

In [ ]:
library(readr) # read_csv(), write_csv()
library(dplyr) # rename(), mutate(), filter(), left_join(), anti_join()
library(tidyr) # pivot_longer()


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [ ]:
data_input <- read_csv(
  "raw_data/Primary_Land_Use_Tax_Lot_Output_(PLUTO)_20261118.csv",
  show_col_types = FALSE
)

Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”


In [ ]:
head(data_input, 1)

borough,Tax block,Tax lot,community board,census tract 2010,cb2010,schooldist,council district,postcode,firecomp,⋯,transitzone,geom,basempdate,dcasdate,edesigdate,landmkdate,masdate,polidate,rpaddate,zoningdate
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,⋯,<chr>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
QN,8786,42,413,1579.03,2005,26,23,11001,E251,⋯,Beyond the Greater Transit Zone,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [ ]:
#Create borough folders
dir.create("raw_data/borough_files", recursive = TRUE)

output_folder <- "raw_data/borough_files"


#Create borough split folders
dir.create("raw_data/borough_files/split", recursive = TRUE)

output_folder <- "raw_data/borough_files/spilt"

Warning message in dir.create("raw_data/borough_files", recursive = TRUE):
“'raw_data/borough_files' already exists”
Warning message in dir.create("raw_data/borough_files/split", recursive = TRUE):
“'raw_data/borough_files/split' already exists”


In [ ]:
#Borough check

borough_classes <- unique(data_input$borough)

borough_classes

[1] "QN" "SI" "BK" "MN" "BX"

In [ ]:
# Borough codes to filter
boroughs <- c("QN", "SI", "BK", "MN", "BX")

In [ ]:
#check databases (issues producing code)

getwd()

dir.exists("raw_data")
dir.exists("raw_data/borough_files")
dir.exists("raw_data/borough_files/split")

[1] "/content/drive/MyDrive/land_suitability_project"

[1] TRUE

[1] TRUE

[1] TRUE

In [ ]:
#Create a CSV for each borough with a max number of rows for data size

# Maximum number of rows per output file
rows_per_file <- 50000

# Output folder
output_folder <- "raw_data/borough_files/split"

#creating the csv files
for (borough_code in boroughs) {

  # Filter data for the current borough
  filtered_data <- data_input %>%
    filter(borough == borough_code)

  # Determine how many files are needed
  number_of_files <- ceiling(
    nrow(filtered_data) / rows_per_file)

  # Create letters: a, b, c, etc.
  part_letters <- letters[seq_len(number_of_files)]

  # Write each 50,000-row section
  for (part_number in seq_along(part_letters)) {

    start_row <- ((part_number - 1) * rows_per_file) + 1
    end_row <- min(
      part_number * rows_per_file,
      nrow(filtered_data))

    part_data <- filtered_data[start_row:end_row, ]

    output_file <- file.path(
      output_folder,
      paste0(
        "raw_PLUTO_",
        borough_code,
        "_part_",
        part_letters[part_number],
        ".csv"))

    write_csv(
      part_data,
      output_file)}}
